In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Tối ưu lại SparkSession: Giảm luồng (local[4]), nhường RAM (20g) cho OS
spark = SparkSession.builder \
    .appName("Feast_Data_Preparation_Safe") \
    .master("local[4]") \
    .config("spark.driver.memory", "20g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.shuffle.partitions", "32") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .getOrCreate()

def check_and_add_timestamp_safe(spark, source_path, target_path):
    print(f"Đang kiểm tra: {source_path}")
    df = spark.read.parquet(source_path)
    
    if "event_timestamp" not in df.columns:
        print(f"-> Thiếu 'event_timestamp'. Đang thêm và phân mảnh lại dữ liệu...")
        df = df.withColumn("event_timestamp", F.current_timestamp())
        
        # KEY FIX: Ép Spark chia thành 16 phân mảnh nhỏ trước khi ghi ra GCS
        df.repartition(16).write.mode("overwrite").parquet(target_path)
        
        print(f"-> Đã ghi thành công ra {target_path}")
        df_result = spark.read.parquet(target_path)
    else:
        print(f"-> Đã có 'event_timestamp'.")
        df_result = df
        
    df_result.printSchema()
    return df_result

base_path = "gs://amazon-reviews-lakehouse-warehouse/warehouse"

print("--- ITEM FEATURES ---")
df_item = check_and_add_timestamp_safe(
    spark, 
    source_path=f"{base_path}/gold/item_features/data", 
    target_path=f"{base_path}/gold/item_features_v2"
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/14 18:09:59 INFO SparkEnv: Registering MapOutputTracker
26/06/14 18:09:59 INFO SparkEnv: Registering BlockManagerMaster
26/06/14 18:09:59 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/06/14 18:09:59 INFO SparkEnv: Registering OutputCommitCoordinator


--- ITEM FEATURES ---
Đang kiểm tra: gs://amazon-reviews-lakehouse-warehouse/warehouse/gold/item_features/data


-> Thiếu 'event_timestamp'. Đang thêm và phân mảnh lại dữ liệu...


-> Đã ghi thành công ra gs://amazon-reviews-lakehouse-warehouse/warehouse/gold/item_features_v2
root
 |-- item_id: string (nullable = true)
 |-- review_count: long (nullable = true)
 |-- avg_rating: double (nullable = true)
 |-- first_review_time: long (nullable = true)
 |-- last_review_time: long (nullable = true)
 |-- item_text_context: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)

